# SynLess CLIP/CUB pilot: LESS + BIDS selection of SCSS negatives
**CLIP-only notebook.** For the separate Qwen learned-generator pilot, open [the LLM notebook](https://colab.research.google.com/github/akashm776/SynLess/blob/main/colabs/SynLess_LLM_A100.ipynb). This notebook does not run LLM training.
Select **Runtime → Change runtime type → A100 GPU**. Start from a fresh runtime. This pilot uses three seeds and two model states. It downloads CLIP and CUB, scores OT/uniform/hardest-real candidates, and tests matched continuations.

CPU tests and toy runs validate the implementation. A completed three-seed GPU pilot is documented in [pilot results](https://github.com/akashm776/SynLess/blob/main/docs/CLIP_PILOT_RESULTS.md). The two retrieval directions are the BIDS target objectives, not quotas over generator types. See the [CLIP guide](https://github.com/akashm776/SynLess/blob/main/docs/CLIP_PILOT.md) for methodological differences from LESS/BIDS.

Results/checkpoints are saved to Drive. Rerun the launch cell after a disconnect. Keep the same code revision and configuration to resume. Do not launch two sessions into the same output folder.

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = Path('/content/SynLess')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/akashm776/SynLess.git', str(REPO)], check=True)
CODE_REF = ''  # To resume on a new runtime, paste the commit printed below.
if CODE_REF:
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', CODE_REF], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Code revision:', COMMIT)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.9.1', 'torchvision==0.24.1', 'torchaudio==2.9.1', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO) + '[clip,test]', 'numpy==2.3.5', 'Pillow==12.0.0'], check=True)
subprocess.run([sys.executable, '-c', "import torch; print(torch.__version__); assert torch.cuda.is_available(), 'Select a GPU runtime'; print(torch.cuda.get_device_name()); assert 'A100' in torch.cuda.get_device_name(), 'Select an A100 runtime'"], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO, check=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/SynLess/runs/pilot_v1')
RUN.mkdir(parents=True, exist_ok=True)
revision_file = RUN / 'code_revision.txt'
if revision_file.exists() and revision_file.read_text().strip() != COMMIT:
    raise RuntimeError('Code revision differs from this run. Set CODE_REF to ' + revision_file.read_text().strip() + ', or use a new RUN directory.')
revision_file.write_text(COMMIT + '\n')
command = [sys.executable, '-u', '-m', 'synless.cli', 'run', '--config', str(REPO / 'configs/colab_a100.json'), '--output', str(RUN)]
print('Output:', RUN)
with (RUN / 'console.log').open('a', buffering=1) as log:
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
            log.write(line)
        returncode = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
    if returncode:
        raise RuntimeError(f'Experiment exited with {returncode}; see console.log. Completed stages are resumable.')

In [ ]:
from IPython.display import display, Markdown
subprocess.run([sys.executable, '-m', 'synless.cli', 'summarize', '--output', str(RUN)], cwd=REPO, check=True)
display(Markdown((RUN / 'REPORT.md').read_text()))
print('For analysis, share REPORT.md, summary.json, and the per-state selection_summary.json / bids_diagnostics.json files.')